# Part 3 — When the source forgets

Part 2 closed on exactly this problem: `fct_orders` keeps only the latest
version of every row, because the source keeps only the latest version too.

It is worse for customers. `raw.customers` holds one row per customer, and
the source overwrites it in place. When a company moves state or gets a new
credit limit, the old value simply stops existing. Most operational systems
work this way.

So `fct_orders` can tell you that Coastal Traders placed an order in June, and
`dim_customers` can tell you they are in New South Wales today. Neither can
tell you where they were **when the order was placed**, and that is the
question every revenue-by-region report is really asking.

dbt snapshots are the tool for this. They also have more sharp edges than any
other dbt feature, enough that dbt Labs opened
[a discussion of their own](https://github.com/dbt-labs/dbt-core/discussions/7018)
to enumerate them.

**What you'll see**

- what a snapshot records, and the two strategies for deciding what changed
- both strategies missing a real change, each blind in a different place
- a deleted customer coming back, and breaking history even with everything
  configured correctly
- the payoff: every order joined to the customer as they were on the day it
  was placed

## Setup

Reset to the initial state. The snapshot runs as part of `dbt build`.

In [1]:
!reset

... restoring raw seed data and dropping dbt schemas


✓ lab restored to its starting state


In [2]:
import pandas as pd

from lab import artifacts, scratch, shop
from lab.runner import dbt, show


def versions(customer_id: int, table: str = "snapshots.customers_snapshot") -> pd.DataFrame:
    """Every version a snapshot holds for one customer, oldest first.
    dbt_is_deleted only exists when hard_deletes asks for it (which becomes
    the point of scenario 4), so select it only where it is present."""
    schema, name = table.split(".")
    has_tombstones = shop.scalar(
        "SELECT COUNT(*) FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = ? "
        "AND TABLE_NAME = ? AND COLUMN_NAME = 'dbt_is_deleted'", (schema, name))
    extra = ", dbt_is_deleted" if has_tombstones else ""
    return shop.df(f"""
        SELECT customer_id, company_name, state, segment, credit_limit_cents,
               dbt_valid_from, dbt_valid_to{extra}
        FROM {table} WHERE customer_id = {customer_id} ORDER BY dbt_valid_from
    """)


result = dbt("build", quiet=True)
assert result.success
m = artifacts.manifest()
print(f"✓ project built · {len(m['nodes']) + len(m['unit_tests'])} nodes green")

✓ project built · 41 nodes green


## What a snapshot records

The source keeps one row per customer. The snapshot keeps one row per
*version* of that customer, each stamped with the interval during which it was
true. This shape is known in data warehousing as a type-2 slowly changing
dimension (**SCD2**):

![what a snapshot records](img/scd2.svg)

Here is the configuration that produces it. Note that the snapshot 
reads the *source* directly. This is dbt's recommendation, so history is
captured before any transformation has a chance to distort it.

In [3]:
show("snapshots/customers_snapshot.yml")

# SCD2 history for a source that overwrites itself. Each run compares what
# the source says now against the latest known version of each row; changed
# rows get a new version, superseded ones get a dbt_valid_to.
#
# strategy: timestamp trusts the source's updated_at completely. Cheap, and
# exactly as reliable as upstream's discipline. Notebook 03 tests that trust.
#
# hard_deletes: new_record. When a customer vanishes upstream, close their
# last version and append a tombstone row (dbt_is_deleted = 'True'). The
# default ('ignore') would keep deleted customers alive forever.
snapshots:
  - name: customers_snapshot
    relation: source('shop', 'customers')
    config:
      unique_key: customer_id
      strategy: timestamp
      updated_at: updated_at
      hard_deletes: new_record



In [4]:
# The five columns dbt adds are easier to believe when you can see them.
shop.df("""
    SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'snapshots' AND TABLE_NAME = 'customers_snapshot'
      AND COLUMN_NAME LIKE 'dbt[_]%'
    ORDER BY COLUMN_NAME
""")

,COLUMN_NAME,DATA_TYPE
0,dbt_is_deleted,varchar
1,dbt_scd_id,varchar
2,dbt_updated_at,datetime2
3,dbt_valid_from,datetime2
4,dbt_valid_to,datetime2


In [5]:
shop.df("""
    SELECT customer_id, company_name, state, segment,
           dbt_valid_from, dbt_valid_to, dbt_is_deleted
    FROM snapshots.customers_snapshot ORDER BY customer_id
""")

,customer_id,company_name,state,segment,dbt_valid_from,dbt_valid_to,dbt_is_deleted
0,1,Acme Pty Ltd,NSW,wholesale,2026-06-02 08:00:00,None,False
1,2,Bunbury Hardware,WA,retail,2026-06-02 08:00:00,None,False
2,3,Coastal Traders,QLD,wholesale,2026-06-10 09:30:00,None,False
3,4,Dandenong Motors,VIC,trade,2026-06-02 08:00:00,None,False
4,5,Esperance Fisheries,WA,wholesale,2026-06-18 14:15:00,None,False
5,6,Fremantle Freight,WA,trade,2026-06-02 08:00:00,None,False
6,7,Geelong Textiles,VIC,retail,2026-06-25 11:45:00,None,False
7,8,Hobart Provedores,TAS,retail,2026-06-02 08:00:00,None,False


Eight customers in, eight versions out. Every `dbt_valid_to` is `NULL`, which
is how SCD2 spells "this is still true".

dbt added five bookkeeping columns. `dbt_valid_from` and `dbt_valid_to` bound
the interval. `dbt_is_deleted` marks a tombstone. `dbt_scd_id` and
`dbt_updated_at` are merge plumbing you can ignore.

**The one decision that matters** is `strategy`. The committed snapshot uses
`timestamp`, which trusts the source's own `updated_at` column: if that
column moved, the row changed. It is cheap, and it is exactly as reliable as
upstream's discipline.

dbt's other option is `check`, which ignores timestamps and compares the
values in a named list of columns. Build that variant alongside, so scenarios
1 to 4 can run the two side by side:

In [6]:
scratch.snapshot_yml("customers_snapshot_check", """
snapshots:
  - name: customers_snapshot_check
    relation: source('shop', 'customers')
    config:
      unique_key: customer_id
      strategy: check
      check_cols: [company_name, state, segment]
""")

result = dbt("snapshot", "--select", "customers_snapshot_check", quiet=True)
assert result.success
assert shop.scalar("SELECT COUNT(*) FROM snapshots.customers_snapshot_check") == 8
print("✓ check-strategy variant baselined: 8 customers")

✓ check-strategy variant baselined: 8 customers


Note what that variant does *not* set: `hard_deletes`. It runs dbt's default,
and scenario 4 is where that matters.

---

## Scenario 1 · A customer moves

Coastal Traders relocates from Queensland to New South Wales. The source
system does everything right: it writes the new state and bumps `updated_at`.

This first run is left loud, once, so you can see both snapshots fire; every
later run goes quiet.

In [7]:
shop.update_customer(3, updated_at="2026-07-30T09:00:00", state="NSW")

result = dbt("snapshot", "--threads", "1")
assert result.success
versions(3)

12:42:44  Running with dbt=1.12.0


12:42:44  Registered adapter: sqlserver=1.10.1


12:42:45  Found 8 models, 30 data tests, 1 seed, 2 snapshots, 4 sources, 537 macros, 1 unit test


12:42:45  


12:42:45  Concurrency: 1 threads (target='dev')


...

12:42:46  


12:42:46  Completed successfully


12:42:46  


12:42:46  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=2


,customer_id,company_name,state,segment,credit_limit_cents,dbt_valid_from,dbt_valid_to,dbt_is_deleted
0,3,Coastal Traders,QLD,wholesale,3500000,2026-06-10 09:30:00,2026-07-30 09:00:00,False
1,3,Coastal Traders,NSW,wholesale,3500000,2026-07-30 09:00:00,NaT,False


In [8]:
v3 = versions(3)
assert len(v3) == 2
assert list(v3["state"]) == ["QLD", "NSW"]
assert v3.loc[0, "dbt_valid_to"] == v3.loc[1, "dbt_valid_from"] == pd.Timestamp("2026-07-30 09:00:00")
assert pd.isna(v3.loc[1, "dbt_valid_to"])
print("✓ QLD ends at the exact instant NSW begins")

✓ QLD ends at the exact instant NSW begins


Two versions, and the boundary is seamless: the old row closes at the exact
moment the new one opens.

Notice *which* moment that is: 30 July 09:00, the value from the source's
`updated_at`. History records when the move **happened**, not when dbt
happened to notice. That is the advantage of the timestamp strategy, and the
next scenario is the price of it.

---

## Scenario 2 · The update nobody stamped

The segment for Bunbury Hardware was mis-entered as `retail`. Someone fixes it to `trade` with a
manual `UPDATE`, the kind that forgets to touch `updated_at`.

Data-entry corrections, backfills and ORM bypasses do this constantly, and the
timestamp strategy's entire worldview rests on that column moving.

In [9]:
shop.update_customer(2, segment="trade")  # deliberately no updated_at bump

result = dbt("snapshot", "--select", "customers_snapshot customers_snapshot_check",
             "--threads", "1", quiet=True)
assert result.success

pd.concat([
    versions(2).assign(snapshot="timestamp (committed)"),
    versions(2, "snapshots.customers_snapshot_check").assign(snapshot="check variant"),
]).set_index("snapshot")

,customer_id,company_name,state,segment,credit_limit_cents,dbt_valid_from,dbt_valid_to,dbt_is_deleted
snapshot,,,,,,,,
timestamp (committed),2,Bunbury Hardware,WA,retail,1500000,2026-06-02 08:00:00.000000,None,False
check variant,2,Bunbury Hardware,WA,retail,1500000,2026-08-03 12:42:44.825125,2026-08-03 12:42:47.196896,NaN
check variant,2,Bunbury Hardware,WA,trade,1500000,2026-08-03 12:42:47.196896,NaT,NaN


### Verdict: the committed snapshot missed it, silently

One version, still `retail`. As far as recorded history is concerned the fix
never happened, and nothing will ever say otherwise.

The `check` variant compared actual values, saw `segment` differ, and wrote a
second version. With one honest asterisk: its `dbt_valid_from` is *processing*
time (when dbt noticed), because a value comparison has no idea when the
change really occurred.

In [10]:
v2_ts = versions(2)
v2_ck = versions(2, "snapshots.customers_snapshot_check")
assert len(v2_ts) == 1 and v2_ts.loc[0, "segment"] == "retail"      # missed
assert len(v2_ck) == 2 and list(v2_ck["segment"]) == ["retail", "trade"]  # caught
print("✓ timestamp missed the fix · check caught it")

✓ timestamp missed the fix · check caught it


---

## Scenario 3 · The column nobody watched

So `check` is simply better?

Esperance Fisheries has its credit limit raised from $25,000 to $40,000,
properly, with `updated_at` bumped. But `credit_limit_cents` is not in the
variant's `check_cols` list.

In [11]:
shop.update_customer(5, updated_at="2026-07-30T10:00:00", credit_limit_cents=4000000)

result = dbt("snapshot", "--select", "customers_snapshot customers_snapshot_check",
             "--threads", "1", quiet=True)
assert result.success

pd.concat([
    versions(5).assign(snapshot="timestamp (committed)"),
    versions(5, "snapshots.customers_snapshot_check").assign(snapshot="check variant"),
]).set_index("snapshot")

,customer_id,company_name,state,segment,credit_limit_cents,dbt_valid_from,dbt_valid_to,dbt_is_deleted
snapshot,,,,,,,,
timestamp (committed),5,Esperance Fisheries,WA,wholesale,2500000,2026-06-18 14:15:00.000000,2026-07-30 10:00:00,False
timestamp (committed),5,Esperance Fisheries,WA,wholesale,4000000,2026-07-30 10:00:00.000000,NaT,False
check variant,5,Esperance Fisheries,WA,wholesale,2500000,2026-08-03 12:42:44.825125,None,NaN


### Verdict: check is exactly as good as its column list

The mirror image of scenario 2. The committed snapshot caught the change,
because the clock moved and so it looked. The `check` variant, watching
`company_name`, `state` and `segment`, never glanced at the credit limit, and
its current row now understates it by $15,000.

A column added to the source next quarter would be invisible the same way,
with even less ceremony.

**Neither strategy dominates.** `timestamp` trusts upstream's discipline;
`check` trusts your enumeration of what matters. Both are trust. Pick the one
whose failure you would rather explain, and put a test on the thing you cannot
afford to lose, which is where part 4 picks up.

In [12]:
v5_ts = versions(5)
v5_ck = versions(5, "snapshots.customers_snapshot_check")
assert len(v5_ts) == 2 and list(v5_ts["credit_limit_cents"]) == [2500000, 4000000]
assert len(v5_ck) == 1 and v5_ck.loc[0, "credit_limit_cents"] == 2500000  # stale
print("✓ check missed what timestamp caught · each has a blind side")

✓ check missed what timestamp caught · each has a blind side


---

## Scenario 4 · A customer is deleted

Hobart Provedores churns, and the source does what operational systems do:
`DELETE`. No tombstone, no event. The row is simply not in the next extract.

For most of dbt's history the only answer to this was *nothing*: the
snapshot keeps the last version open forever, and a churned customer stays
indistinguishable from a live one. The committed snapshot opts into
`hard_deletes: new_record`. The check variant, which never set it, is still on
that default.

In [13]:
shop.delete_customer(8)

result = dbt("snapshot", "--select", "customers_snapshot customers_snapshot_check",
             "--threads", "1", quiet=True)
assert result.success

pd.concat([
    versions(8).assign(snapshot="timestamp + new_record (committed)"),
    versions(8, "snapshots.customers_snapshot_check").assign(snapshot="check + ignore (default)"),
]).set_index("snapshot")

,customer_id,company_name,state,segment,credit_limit_cents,dbt_valid_from,dbt_valid_to,dbt_is_deleted
snapshot,,,,,,,,
timestamp + new_record (committed),8,Hobart Provedores,TAS,retail,800000,2026-06-02 08:00:00.000000,2026-08-03 12:42:49,False
timestamp + new_record (committed),8,Hobart Provedores,TAS,retail,800000,2026-08-03 12:42:49.000000,NaT,True
check + ignore (default),8,Hobart Provedores,TAS,retail,800000,2026-08-03 12:42:44.825125,None,NaN


### Verdict: the default keeps the dead alive

The committed snapshot closed the last real version and appended a
**tombstone**: `dbt_is_deleted = 'True'`, carrying the last known values, with
`valid_from` stamped at processing time. A deletion has no `updated_at`, so
noticing-time is the only clock available.

Anything downstream can now distinguish three states: current, superseded,
and gone.

The default-config variant did nothing at all. Its latest version for customer
#8 still reads as live. That customer will sit in every "active customers"
query indefinitely, which is how marts drift from reality one quiet delete at
a time.

In [14]:
v8 = versions(8)
assert len(v8) == 2
assert list(v8["dbt_is_deleted"]) == ["False", "True"]
assert v8.loc[0, "dbt_valid_to"] == v8.loc[1, "dbt_valid_from"]      # chain intact
v8_ck = versions(8, "snapshots.customers_snapshot_check")
assert len(v8_ck) == 1 and pd.isna(v8_ck.loc[0, "dbt_valid_to"])     # still "alive"
print("✓ committed: closed + tombstoned · default: still alive")

✓ committed: closed + tombstoned · default: still alive


---

## Scenario 5 · The customer comes back

Hobart Provedores re-signs, under its old customer id, upgraded to
`wholesale`. The source inserts the row; the loader lands it. Routine.

This is the scenario that gets the *well-configured* snapshot.

In [15]:
shop.insert_customer(8, "Hobart Provedores", "TAS", "wholesale", 1200000,
                     updated_at="2026-07-31T09:00:00")

result = dbt("snapshot", "--select", "customers_snapshot", "--threads", "1", quiet=True)
assert result.success
versions(8)

,customer_id,company_name,state,segment,credit_limit_cents,dbt_valid_from,dbt_valid_to,dbt_is_deleted
0,8,Hobart Provedores,TAS,retail,800000,2026-06-02 08:00:00,2026-08-03 12:42:49,False
1,8,Hobart Provedores,TAS,wholesale,1200000,2026-07-31 09:00:00,NaT,False
2,8,Hobart Provedores,TAS,retail,800000,2026-08-03 12:42:49,NaT,True


### Verdict: two rows are now open at once

Look at `dbt_valid_to`. The tombstone was never closed, and the revived
version opened with `valid_from` taken from its `updated_at`, which is
*earlier* than the tombstone's processing-time stamp.

Customer #8 now has two open-ended versions whose intervals overlap. Recorded
history claims the customer was simultaneously deleted and a wholesale
account.

This is not an adapter quirk, and it is not really a bug either. It falls
straight out of part 2's two clocks: the deletion could only be stamped with
dbt's clock, the revival carries the source's, and nothing can put those two
in order.

In [16]:
open_rows = shop.scalar("""
    SELECT COUNT(*) FROM snapshots.customers_snapshot
    WHERE customer_id = 8 AND dbt_valid_to IS NULL
""")
assert open_rows == 2  # tombstone AND revived version, both "current"

fanout = shop.scalar("""
    SELECT COUNT(*) FROM snapshots.customers_snapshot
    WHERE customer_id = 8 AND dbt_valid_to IS NULL AND SYSDATETIME() >= dbt_valid_from
""")
assert fanout == 2  # a naive point-in-time lookup returns two rows
print("✓ two open versions for one key · the run went green anyway")

✓ two open versions for one key · the run went green anyway


**The defence is downstream discipline, not a config value.** Two things are
needed: a dimension built from the snapshot must filter tombstones, and the
invariant *one open row per key* has to be pinned down by a test, because
nothing in dbt will notice it for you.

Write that test now, so the failure is caught rather than described:

In [17]:
scratch.model("assert_one_open_version_per_customer", """
{{ config(materialized='view', schema='scratch') }}
-- Returns a row per customer with more than one open version. Zero rows is
-- the invariant holding; anything else is a broken validity chain.
select customer_id, count(*) as open_versions
from {{ ref('customers_snapshot') }}
where dbt_valid_to is null
group by customer_id
having count(*) > 1
""")

assert dbt("run", "--select", "assert_one_open_version_per_customer", quiet=True).success
broken = shop.df("SELECT * FROM scratch.assert_one_open_version_per_customer")
assert len(broken) == 1 and broken.loc[0, "customer_id"] == 8
print("✓ the invariant now fails loudly instead of silently · customer #8, 2 open versions")
broken

✓ the invariant now fails loudly instead of silently · customer #8, 2 open versions


,customer_id,open_versions
0,8,2


Written as a *singular test* (a hand-written SQL test in `tests/`), that
query turns scenario 5 from silent into loud: `dbt build` would stop rather
than hand an analyst a fan-out. Part 4 is about exactly that move: turning
things you have learned the hard way into checks that run.

---

## Putting the history to work

History that only accumulates is a liability. The payoff is point-in-time
truth: joining each order to the customer **as they were on the day it was
placed**.

Build a dimension over the snapshot, filtering tombstones and renaming dbt's
bookkeeping into business terms:

In [18]:
scratch.model("dim_customers_history", """
{{ config(materialized='table', schema='scratch') }}

select
    customer_id,
    company_name,
    state as state_code,
    segment,
    dbt_valid_from as valid_from,
    dbt_valid_to   as valid_to
from {{ ref('customers_snapshot') }}
where dbt_is_deleted = 'False'
""")

result = dbt("run", "--select", "dim_customers_history", quiet=True)
assert result.success
assert shop.scalar(
    "SELECT COUNT(*) FROM scratch.dim_customers_history "
    "WHERE customer_id = 8 AND valid_to IS NULL") == 1
print("✓ tombstone filtered · customer #8 back to one current row")

✓ tombstone filtered · customer #8 back to one current row


Coastal Traders moved from Queensland to New South Wales in scenario 1. Land
one more order *after* the move, let the incremental mart from part 2 pick it
up, and ask where each order belongs:

In [19]:
shop.place_order(order_id=32, customer_id=3, product_id=2, quantity=5,
                 unit_price_cents=3450, placed_at="2026-07-30T14:00:00")
result = dbt("run", "--select", "fct_orders", quiet=True)
assert result.success

asof = shop.df("""
    SELECT o.order_id, o.placed_at, c.company_name, c.state_code
    FROM marts.fct_orders o
    JOIN scratch.dim_customers_history c
      ON o.customer_id = c.customer_id
     AND o.placed_at >= c.valid_from
     AND o.placed_at < COALESCE(c.valid_to, '9999-12-31')
    WHERE o.customer_id = 3
    ORDER BY o.placed_at
""")
asof

,order_id,placed_at,company_name,state_code
0,2,2026-07-01 11:02:00,Coastal Traders,QLD
1,11,2026-07-09 09:05:00,Coastal Traders,QLD
2,19,2026-07-16 10:18:00,Coastal Traders,QLD
3,27,2026-07-25 13:29:00,Coastal Traders,QLD
4,32,2026-07-30 14:00:00,Coastal Traders,NSW


### Verdict: each order attributed to its own era

Same customer, same query. Orders placed before 30 July report Queensland; the
order after the move reports New South Wales.

A current-state dimension would have quietly re-filed years of Queensland
revenue under New South Wales the moment the source updated. This is the
difference between history and *the present, repeated*.

In [20]:
assert set(asof[asof["placed_at"] < "2026-07-30"]["state_code"]) == {"QLD"}
assert set(asof[asof["placed_at"] >= "2026-07-30"]["state_code"]) == {"NSW"}
assert len(asof) == asof["order_id"].nunique()  # the join did not fan out
print("✓ pre-move orders → QLD · post-move order → NSW · no fan-out")

✓ pre-move orders → QLD · post-move order → NSW · no fan-out


---

## What the scenarios add up to

| lesson | in one line |
|---|---|
| `timestamp` strategy | exactly as reliable as upstream's `updated_at` discipline |
| `check` strategy | exactly as complete as your column list, and stamped with dbt's clock, not the world's |
| `hard_deletes: new_record` | the non-default that keeps deletions visible |
| a returning key | breaks the validity chain even when everything is configured right |
| point-in-time (as-of) joins | the payoff that makes operating all of the above worthwhile |

One more point worth stating plainly: **snapshot mistakes are not
recoverable.** An incremental model can be rebuilt with `--full-refresh`,
because its source of truth still exists. A snapshot *is* the source of truth
for history. Get the strategy wrong for six months and the versions you failed
to capture do not exist anywhere to be recaptured.

Every failure in this notebook and the last one went green. This raises the next question:
how much of it would dbt's own testing tools have caught?

*Next: [proving it's right →](04-testing-and-contracts.ipynb)*